# Cargar las librerias

In [6]:
import tensorflow as tf

from tensorflow.keras import layers, models
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Cargar los datos

In [2]:
val_glove = np.load('../datos/encuestav1_gloVe_Values.npy', allow_pickle=True)
val_glove.shape

(460,)

In [3]:
data = pd.read_csv('../datos/train_poll_v1s1.csv')
labels = data['ai']
labels = np.array(labels)
labels.shape

(460,)

In [ ]:
# Split data into training and validation sets (80% train, 20% validation)
max_timesteps = 8

# Step 1: Pad the sequences
X_padded = pad_sequences(val_glove, maxlen=max_timesteps, dtype='float32', padding='post', truncating='post')

# Step 2: Create mask (non-zero embeddings)
mask = (X_padded.sum(axis=-1) != 0)  # shape: (batch_size, max_timesteps), bool
# Step 3: Split into train/val sets
X_train, X_val, y_train, y_val, mask_train, mask_val = train_test_split(
    X_padded, labels, mask, test_size=0.2, random_state=42
)


[[ True  True  True ... False False False]
 [ True  True  True ... False False False]
 [ True  True  True ... False False False]
 ...
 [ True False False ... False False False]
 [ True False False ... False False False]
 [ True  True  True ... False False False]]


# Crear el modelo y entrenar

In [10]:
# Assume val_glove is a list/array of variable-length sequences of GloVe embeddings
# Each embedding is a (embedding_dim,) vector

embedding_dim = 300
max_timesteps = 8

# Pad sequences to same length
X_padded = pad_sequences(val_glove, maxlen=max_timesteps, dtype='float32', padding='post', truncating='post')

# Create boolean mask: True where the sequence is non-zero (not padding)
mask = (X_padded.sum(axis=-1) != 0)

# Split data
X_train, X_val, y_train, y_val, mask_train, mask_val = train_test_split(
    X_padded, labels, mask, test_size=0.2, random_state=42
)


In [13]:
def create_transformer_binary_classifier(embedding_dim, num_heads, ff_dim):
    input_embeddings = tf.keras.Input(shape=(None, embedding_dim), name='input_embeddings')
    input_mask = tf.keras.Input(shape=(None,), dtype=tf.bool, name='input_mask')

        # Expand dims to shape (batch_size, 1, 1, seq_len) — required by MultiHeadAttention
    expanded_mask = layers.Lambda(lambda m: tf.cast(tf.expand_dims(tf.expand_dims(m, 1), 1), tf.float32))(input_mask)

    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=embedding_dim,
        dropout=0.1
    )(input_embeddings, input_embeddings, attention_mask=expanded_mask)


    # Add & Norm
    x = layers.Add()([attention_output, input_embeddings])
    x = layers.LayerNormalization()(x)

    # Feed-forward network
    ffn_output = layers.Dense(ff_dim, activation='relu')(x)
    ffn_output = layers.Dense(embedding_dim)(ffn_output)

    # Add & Norm
    x = layers.Add()([ffn_output, x])
    x = layers.LayerNormalization()(x)

    # Global pooling (can replace with first-token approach if desired)
    pooled_output = layers.GlobalAveragePooling1D()(x)

    # Final classification head
    output = layers.Dense(1, activation='sigmoid')(pooled_output)

    model = models.Model(inputs=[input_embeddings, input_mask], outputs=output)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


In [14]:
model = create_transformer_binary_classifier(
    embedding_dim=embedding_dim,
    num_heads=4,
    ff_dim=128
)

model.summary()

model.fit(
    {'input_embeddings': X_train, 'input_mask': mask_train},
    y_train,
    validation_data=({'input_embeddings': X_val, 'input_mask': mask_val}, y_val),
    epochs=5,
    batch_size=16
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_mask          │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_embeddings    │ (None, None, 300) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1, 1,      │          0 │ input_mask[0][0]  │
│                     │ None)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 300) │  1,443,900 │ input_embeddings… │
│ (MultiHeadAttentio… │                   │            │ input_embeddings… │
│                     │                   │            │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 300) │          0 │ multi_head_atten… │
│                     │                   │            │ input_embeddings… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 300) │        600 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 128) │     38,528 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 300) │     38,700 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 300) │          0 │ dense_1[0][0],    │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 300) │        600 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 300)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │        301 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,522,629 (5.81 MB)

 Trainable params: 1,522,629 (5.81 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 10s 111ms/step - accuracy: 0.6244 - loss: 1.2565 - val_accuracy: 0.9130 - val_loss: 0.3803
Epoch 2/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 0.9004 - loss: 0.2957 - val_accuracy: 0.9130 - val_loss: 0.3491
Epoch 3/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 0.9398 - loss: 0.1471 - val_accuracy: 0.9130 - val_loss: 0.3637
Epoch 4/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - accuracy: 0.9551 - loss: 0.1036 - val_accuracy: 0.9022 - val_loss: 0.3299
Epoch 5/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - accuracy: 0.9589 - loss: 0.0899 - val_accuracy: 0.9348 - val_loss: 0.3618


In [23]:


class TransformerEncoderLayer(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super().__init__()
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            layers.Dense(ff_dim, activation="relu"),
            layers.Dense(embed_dim),
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False, mask=None):
        attn_output = self.att(inputs, inputs, attention_mask=mask)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)

        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Main model
embedding_dim = 300
num_heads = 2
ff_dim = 128

# Input layer allows variable sequence length
inputs = tf.keras.Input(shape=(None, embedding_dim), name="inputs")

# Optional mask for padding (zeros = pad)
mask_input = tf.keras.Input(shape=(None,), dtype=tf.bool, name="mask")

# Transformer with attention mask
x = TransformerEncoderLayer(embedding_dim, num_heads, ff_dim)(inputs, mask=mask_input)

# Output
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(1, activation='sigmoid')(x)

# Model
model = tf.keras.Model(inputs=[inputs, mask_input], outputs=x)
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, None, 300) │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mask (InputLayer)   │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_encode… │ (None, None, 300) │    800,528 │ inputs[0][0],     │
│ (TransformerEncode… │                   │            │ mask[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 300)       │          0 │ transformer_enco… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │     19,264 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         65 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 819,857 (3.13 MB)

 Trainable params: 819,857 (3.13 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# Mask creation (to handle padding for variable length sequences)
mask_train = np.any(X_train != 0, axis=-1)
mask_val = np.any(X_val != 0, axis=-1)
mask_val

C:\Users\angel\AppData\Local\Temp\ipykernel_8308\4047965678.py:2: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  mask_train = np.any(X_train != 0, axis=-1)
C:\Users\angel\AppData\Local\Temp\ipykernel_8308\4047965678.py:3: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  mask_val = np.any(X_val != 0, axis=-1)


True

In [ ]:

# Train the model
model.fit([X_train, mask_train], y_train, batch_size=32, epochs=5, validation_data=([X_val, mask_val], y_val))

ValueError: Failed to convert a NumPy array to a Tensor (Unsupported object type numpy.ndarray).

In [33]:
import numpy as np

# Example of sequences (4 tokens per sequence)
X_train = [
    [0.5, 0.3, 0, 0],  # sequence 1 with padding at 2 and 3
    [0.1, 0.4, 0.2, 0]  # sequence 2 with padding at 3
]

# Create a mask where True = valid tokens, False = padding tokens
mask_train = X_train != 0  # Check for non-zero values
print(mask_train)


True
